# sgd-vanilla-from-scratch — ex2: run SGD on a quadratic; verify convergence + monotone loss decrease

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `sgd-vanilla-from-scratch`. Running the final beacon cell reports progress against the `Optimizer: SGD vanilla from scratch` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: SGD vanilla from scratch` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sgd-vanilla-from-scratch`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sgd-vanilla-from-scratch"
DD_SUBTOPIC = "Optimizer: SGD vanilla from scratch"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## SGD convergence on a quadratic — deepening

Ex1 verified a single step. Ex2 verifies BEHAVIOR over many steps. On the convex quadratic `L(w) = 0.5 * (w - w*)^2`, the gradient is `dL/dw = w - w*` and the SGD recursion is:

```
w_{k+1} = w_k - lr * (w_k - w*) = (1 - lr) * w_k + lr * w*
```

For `0 < lr < 2`, the iterates contract toward `w*` at rate `|1 - lr|^k`. Two consequences:

1. `w_k -> w*` (provided `lr < 2`).
2. `L(w_k)` is **monotonically decreasing** in `k` (for `0 < lr <= 1`; for `lr > 1` it can oscillate but still converge if `lr < 2`).

These two facts are how you sanity-check ANY new optimizer: run it on a quadratic, check (a) the optimum is reached, (b) the loss decreases each step at small lr.

### Exercise 2 — run SGD on a quadratic; verify convergence + monotone loss decrease

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply vanilla SGD over many iterations to a 1-D quadratic loss, producing a sequence of weight + loss values that converge to the known minimum with monotonically decreasing loss.
> Keywords: sgd, convergence, quadratic, monotone-loss, multi-step
> ```

**KCs targeted:** `sgd-vanilla-from-scratch`, `convergence-on-quadratic`

Implement `sgd_step(params, lr)` (same as ex1) and a driver `run_sgd_on_quadratic(w0, w_star, lr, n_steps)` that:

1. Wrap `w0` (a Python float) as a MiniTensor with `requires_grad=True`.
2. For `n_steps` iterations:
   - Compute `loss = 0.5 * (w.array.item() - w_star) ** 2` (a Python float — we don't need the autograd graph for this drill).
   - Compute `w.grad = w.array - w_star` (a tensor of shape matching `w.array`).
   - Call `sgd_step([w], lr)`.
   - Append the loss to a `losses` list AFTER the update is NOT needed — we append the loss as it was BEFORE the update, i.e. right after computing it.
3. Return `(final_w_value, losses)` — `(float, list[float])`.

**Logging order.** Record loss BEFORE the step so the list reads `[L(w_0), L(w_1), ..., L(w_{n-1})]`. The test verifies this list is monotonically non-increasing.

**Don't use real autograd.** This drill is about the OPTIMIZER, not the graph. Set `w.grad` directly from the closed-form `w - w*`.

In [ ]:
def sgd_step(params: list, lr: float) -> None:
    """One SGD step (same as ex1)."""
    raise NotImplementedError()


def run_sgd_on_quadratic(w0: float, w_star: float, lr: float, n_steps: int):
    """Return (final_w: float, losses: list[float])."""
    raise NotImplementedError()


def _test_ex2():
    # --- invariant 1: converges to w_star at moderate lr ---
    final, losses = run_sgd_on_quadratic(w0=0.0, w_star=5.0, lr=0.1, n_steps=200)
    assert isinstance(final, float), f'final must be float, got {type(final).__name__}'
    assert len(losses) == 200, f'losses must have n_steps entries, got {len(losses)}'
    assert abs(final - 5.0) < 1e-4, f'should converge to w_star=5, got {final}'

    # --- invariant 2: monotonically non-increasing loss at lr=0.1 (under 1) ---
    for i in range(len(losses) - 1):
        assert losses[i + 1] <= losses[i] + 1e-9, (
            f'loss must be non-increasing at lr=0.1; '
            f'step {i}: {losses[i]} -> step {i+1}: {losses[i+1]}'
        )

    # --- invariant 3: final loss is essentially zero ---
    assert losses[-1] < 1e-8, f'final loss should be ~0, got {losses[-1]}'

    # --- invariant 4: geometric decay rate matches theory ---
    # L(w_k) = 0.5 * |w_k - w*|^2 ; ratio of successive losses = (1-lr)^2 = 0.81 at lr=0.1
    # Compare losses[10] / losses[0]  ~  0.81^10 ~ 0.1216
    ratio = losses[10] / losses[0]
    expected_ratio = (1 - 0.1) ** (2 * 10)
    rel_err = abs(ratio - expected_ratio) / expected_ratio
    assert rel_err < 0.05, (
        f'loss decay rate must match theory; got ratio {ratio:.5f} vs '
        f'expected {expected_ratio:.5f} (rel err {rel_err:.4f})'
    )

    # --- invariant 5: starting at the minimum -> zero loss, zero motion ---
    final_at_min, losses_at_min = run_sgd_on_quadratic(w0=5.0, w_star=5.0, lr=0.1, n_steps=20)
    assert abs(final_at_min - 5.0) < 1e-9, 'starting at minimum: w should not move'
    assert all(L < 1e-12 for L in losses_at_min), 'starting at minimum: every loss must be zero'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def sgd_step(params: list, lr: float) -> None:
    for p in params:
        if p.grad is None:
            continue
        p.array -= lr * p.grad
        p.grad = None


def run_sgd_on_quadratic(w0: float, w_star: float, lr: float, n_steps: int):
    w = MiniTensor(t.tensor([w0]), requires_grad=True)
    losses = []
    for _ in range(n_steps):
        # record loss before the step
        loss = 0.5 * (w.array.item() - w_star) ** 2
        losses.append(loss)
        # closed-form grad of 0.5 * (w - w*)^2 is (w - w*)
        w.grad = w.array - w_star
        sgd_step([w], lr)
    return float(w.array.item()), losses
```

**Geometric convergence is the fingerprint of SGD on a quadratic.** The recursion `w_{k+1} = (1 - lr) * w_k + lr * w*` is a 1-D linear contraction with rate `|1 - lr|`. Loss decays as `(1 - lr)^{2k}`, which is what test invariant 4 asserts. If you see a different rate in your implementation — likely off by a factor of `lr` somewhere — this is the test that catches it.

**`losses` records BEFORE the step.** So `losses[0]` is the initial loss (largest), `losses[-1]` is the loss at the second-to-last iterate. After the final step, `final_w` itself has converged but we don't append a final loss. (Pick a convention and stick to it; this matches PyTorch's typical training-loop pattern.)

**Don't use autograd here.** The atom under study is the OPTIMIZER. Wiring `w.grad` directly to the closed-form derivative isolates the test from any autograd bugs.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()